In [4]:
import os
import pandas as pd

# Read the CSV file
csv_file = 'pages.csv'  # Replace with your CSV file name
html_column = 'page_content'  # Replace with the name of the column containing HTML code
output_dir = 'html_pages'  # Directory to save the HTML files

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Load CSV into a DataFrame
df = pd.read_csv(csv_file)

# Iterate over the rows and save each HTML content to a file
for index, row in df.iterrows():
    html_content = row[html_column]  # Extract the HTML code
    
    # Handle missing or non-string values
    if pd.isna(html_content):
        html_content = ""  # Replace NaN or missing values with an empty string
    else:
        html_content = str(html_content)  # Ensure the content is a string
    
    # Create the output file name
    output_file = os.path.join(output_dir, f'page_{index + 1}.html')
    
    # Write the HTML content to the file
    with open(output_file, 'w', encoding='utf-8') as file:
        file.write(html_content)

print(f"HTML files have been saved to the '{output_dir}' directory.")


HTML files have been saved to the 'html_pages' directory.


In [1]:
import pandas as pd
from bs4 import BeautifulSoup

# Load the original CSV file
csv_file = "pages.csv"  # Replace with your actual file path
df = pd.read_csv(csv_file)

# Function to extract text from HTML
def extract_text_from_html(html_content):
    if pd.isnull(html_content):  # Check for missing values
        return ""  # Return empty string if content is null
    soup = BeautifulSoup(html_content, "html.parser")  # Parse HTML
    return soup.get_text(strip=True)  # Extract plain text

# Apply the function to 'page_content' and overwrite or add a column
df['cleaned_text'] = df['page_content'].apply(extract_text_from_html)

# Save the updated DataFrame back to the original CSV file
df.to_csv(csv_file, index=False)

# Print confirmation
print("Cleaned text has been saved back to the original CSV file.")


Cleaned text has been saved back to the original CSV file.


In [3]:
import pandas as pd
from bs4 import BeautifulSoup

# Load the original CSV file
input_csv = "pages.csv"  # Replace with your original CSV file path
output_csv = "cleaned_html.csv"  # Path for the new CSV file

# Load the data
df = pd.read_csv(input_csv)

# Function to extract reviews/blogs from HTML content
def extract_reviews_from_html(html_content):
    if pd.isnull(html_content):  # Check for missing values
        return ""  # Return empty string if content is null
    
    soup = BeautifulSoup(html_content, "html.parser")  # Parse HTML
    
    # Remove unwanted tags like headings, navigation, and other noise
    for tag in soup(["h1", "h2", "h3", "nav", "footer", "header", "aside"]):
        tag.decompose()  # Remove the tag and its content
    
    # Extract desired content: paragraphs or relevant tags
    reviews = []
    for tag in soup.find_all(["p", "article", "div"]):
        text = tag.get_text(strip=True)
        if len(text) > 20:  # Skip short or irrelevant text
            reviews.append(text)
    
    return " ".join(reviews)  # Combine all extracted text into one string

# Apply the function to clean 'page_content'
df['cleaned_text'] = df['page_content'].apply(extract_reviews_from_html)

# Select columns to include in the new CSV
columns_to_save = ['Title', 'Link', 'Description', 'cleaned_text']
df_cleaned = df[columns_to_save]  # Create a new DataFrame with selected columns

# Save the cleaned data to a new CSV file
df_cleaned.to_csv(output_csv, index=False)

# Print confirmation
print(f"Cleaned data has been saved to '{output_csv}' with title, link, description, and cleaned text.")

Cleaned data has been saved to 'cleaned_reviews.csv' with title, link, description, and cleaned text.


In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
from transformers import pipeline
import hashlib
# Load the CSV file
input_csv = "scraped_results.csv"  # Replace with the path to your original file
output_csv = "cleaned_reviews_tf.csv"

# Load a text generation LLM pipeline (open-source, e.g., Hugging Face's Mistral or Falcon model)
# You can replace this model with any available open-source model from Hugging Face
llm = pipeline("text-generation", model="tiiuae/falcon-7b", max_new_tokens=200)

# Function to clean HTML and prepare content
def clean_html_content(html_content):
    if pd.isnull(html_content):  # Handle missing values
        return ""
    soup = BeautifulSoup(html_content, "html.parser")
    # Remove unnecessary tags like headings, navigation, etc.
    for tag in soup(["h1", "h2", "nav", "footer", "header", "aside"]):
        tag.decompose()
    return soup.get_text(strip=True)

# Function to extract reviews using LLM
def extract_reviews_with_llm(content):
    if not content:
        return ""
    # Prompt the LLM to extract reviews/blog-like content
    prompt = f"Extract only the reviews or blog content from the following text:\n{content}"
    response = llm(prompt)
    return response[0]['generated_text']

# Function to deduplicate results using hashing
def deduplicate_texts(texts):
    seen = set()
    unique_texts = []
    for text in texts:
        text_hash = hashlib.md5(text.encode('utf-8')).hexdigest()
        if text_hash not in seen:
            seen.add(text_hash)
            unique_texts.append(text)
    return unique_texts

# Load the data
df = pd.read_csv(input_csv)

# Clean HTML content and extract reviews using LLM
df['cleaned_text'] = df['page_content'].apply(lambda x: extract_reviews_with_llm(clean_html_content(x)))

# Deduplicate the cleaned_text column
df['cleaned_text'] = deduplicate_texts(df['cleaned_text'])

# Select relevant columns for the output
columns_to_save = ['title', 'link', 'description', 'cleaned_text']
df_cleaned = df[columns_to_save]

# Save the cleaned data to a new CSV file
df_cleaned.to_csv(output_csv, index=False)

print(f"Cleaned and deduplicated data has been saved to '{output_csv}'.")


2025-01-09 22:13:27.098549: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-01-09 22:13:27.273857: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1736441007.364694 1974753 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1736441007.389333 1974753 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-09 22:13:27.589733: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [1]:
import torch
print("GPU Available:", torch.cuda.is_available())
print("Device Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

GPU Available: True
Device Name: NVIDIA GeForce RTX 3050 Laptop GPU


In [1]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
import pandas as pd
from bs4 import BeautifulSoup
import shutil
import torch

# Load the CSV file
input_csv = "scraped_results.csv"  # Path to your input CSV
output_csv = "cleaned_reviews_tf.csv"  # Path to save the output CSV

# Load Flan-T5 model and tokenizer
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    device_map="auto"  # Automatically use GPU if available
)

# Create a text-generation pipeline
text_generator = pipeline("text2text-generation", model=model, tokenizer=tokenizer)

# Function to clean HTML content from `page_content`
def clean_html_content(html_content):
    if pd.isnull(html_content):
        return ""
    soup = BeautifulSoup(html_content, "html.parser")
    # Remove unwanted tags like headings, navigation bars, etc.
    for tag in soup(["h1", "h2", "nav", "footer", "header", "aside"]):
        tag.decompose()
    return soup.get_text(strip=True)

# Function to extract reviews using Flan-T5
def extract_reviews_with_flan(content):
    if not content.strip():
        return ""
    # Prompt for extracting reviews or blog-like content
    prompt = f"Extract only the reviews or blog content from the following text:\n{content}"
    result = text_generator(prompt, max_new_tokens=150, num_return_sequences=1)
    return result[0]['generated_text']

torch.cuda.memory_summary(device=None, abbreviated=False)

# Load the CSV into a DataFrame
df = pd.read_csv(input_csv)

# Process in smaller chunks
batch_size = 10
chunks = [df[i:i + batch_size] for i in range(0, df.shape[0], batch_size)]

for i, chunk in enumerate(chunks):
    chunk['cleaned_text'] = chunk['page_content'].apply(
        lambda x: extract_reviews_with_flan(clean_html_content(x))
    )
    # Save each chunk incrementally
    chunk.to_csv(f"cleaned_reviews_part_{i}.csv", index=False)

# Process each row to clean HTML and extract reviews using Flan-T5
df['cleaned_text'] = df['page_content'].apply(lambda x: extract_reviews_with_flan(clean_html_content(x)))

# Deduplicate results
df['cleaned_text'] = df['cleaned_text'].drop_duplicates()

# Select relevant columns for the output CSV
columns_to_save = ['title', 'link', 'description', 'cleaned_text']
df_cleaned = df[columns_to_save]

# Save the cleaned data to a new CSV
df_cleaned.to_csv(output_csv, index=False)

print(f"Cleaned reviews have been saved to '{output_csv}'.")


2024-12-20 20:49:37.899392: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-12-20 20:49:38.029748: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1734707978.083211    8372 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1734707978.098455    8372 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-20 20:49:38.222813: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

OutOfMemoryError: CUDA out of memory. Tried to allocate 306.00 MiB. GPU 0 has a total capacity of 3.80 GiB of which 212.19 MiB is free. Including non-PyTorch memory, this process has 3.58 GiB memory in use. Of the allocated memory 3.41 GiB is allocated by PyTorch, and 86.95 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)